# HAKE-MER — Step 0 baseline (GoEmotions protocol)

**Protocol:** batch 16, LR 5e-5, **4 epochs**, 3 seeds, best val F1-macro checkpoint.

## Phase 1 — DistilBERT only

1. **Runtime → Factory reset** → **GPU**
2. Run cells through **Download DistilBERT zip** (~1–1.5 h on T4)

## Phase 2 — RoBERTa (later)

1. **Uncomment** the RoBERTa training cell
2. Run it, then **Download RoBERTa zip**

In [1]:
import torch
if not torch.cuda.is_available():
    raise RuntimeError("Runtime → Change runtime type → GPU")
print("GPU:", torch.cuda.get_device_name(0))

GPU: NVIDIA A100-SXM4-40GB


In [2]:
import subprocess
from getpass import getpass
from pathlib import Path

REPO, WORKDIR = "khalef-khalil/marii", Path("/content/marii")
PUBLIC_URL = f"https://github.com/{REPO}.git"
TRAIN_FLAGS = "--epochs 4 --batch-size 16 --lr 5e-5 --early-stopping-patience 0"

def clone_repo() -> None:
    if WORKDIR.is_dir():
        subprocess.run(["git", "-C", str(WORKDIR), "fetch", "origin", "main"], check=True)
        subprocess.run(["git", "-C", str(WORKDIR), "reset", "--hard", "origin/main"], check=True)
        return
    r = subprocess.run(["git", "clone", "--depth", "1", PUBLIC_URL, str(WORKDIR)], capture_output=True)
    if r.returncode == 0:
        return
    try:
        from google.colab import userdata
        token = userdata.get("GITHUB_TOKEN")
    except Exception:
        token = getpass("GitHub token: ")
    subprocess.run(["git", "clone", "--depth", "1", f"https://{token}@github.com/{REPO}.git", str(WORKDIR)], check=True)

clone_repo()
%cd {WORKDIR}
!git rev-parse --short HEAD

/content/marii
d5cbf03


In [3]:
!pip install -q -r requirements-train.txt

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.5/44.5 kB 4.1 MB/s eta 0:00:00


### Phase 1 — train DistilBERT

In [4]:
!./run_baseline_campaign.sh --backbone distilbert-base-uncased {TRAIN_FLAGS}

config.json: 100% 483/483 [00:00<00:00, 1.73MB/s]
tokenizer_config.json: 100% 48.0/48.0 [00:00<00:00, 261kB/s]
vocab.txt: 100% 232k/232k [00:00<00:00, 13.3MB/s]
tokenizer.json: 100% 466k/466k [00:00<00:00, 1.13MB/s]
README.md: 100% 9.40k/9.40k [00:00<00:00, 26.5MB/s]

simplified/train-00000-of-00001.parquet: downloading bytes:   0% 13.5k/2.77M [00:01<03:49, 12.0kB/s]
simplified/train-00000-of-00001.parquet: downloading bytes: 100% 2.73M/2.73M [00:01<00:00, 2.11MB/s,  263kB/s  ]
simplified/train-00000-of-00001.parquet: reconstructing file: 100% 2.77M/2.77M [00:01<00:00, 2.14MB/s,  267kB/s  ]

simplified/validation-00000-of-00001.par(…): downloading bytes:   0% 0.00/350k [00:00<?, ?B/s]
simplified/validation-00000-of-00001.par(…): downloading bytes: 100% 346k/346k [00:01<00:00, 324kB/s, 33.6kB/s  ]
simplified/validation-00000-of-00001.par(…): reconstructing file: 100% 350k/350k [00:01<00:00, 328kB/s, 34.0kB/s  ]

simplified/test-00000-of-00001.parquet: downloading bytes:   0% 0.00/347k [

In [5]:
import json
from pathlib import Path

def show_campaign(name: str) -> None:
    path = Path("reference/artifacts") / name
    if not path.is_file():
        print(f"Missing {name}")
        return
    c = json.loads(path.read_text(encoding="utf-8"))
    print(path.name, "protocol:", c.get("protocol", {}))
    print("  epochs logged:", len(c["runs"][0]["history"]))
    for k, b in c["test_aggregate"].items():
        print(f"  {k}: {b['mean']:.4f} ± {b['std']:.4f}")

show_campaign("baseline_plm_distilbert_base_uncased_campaign.json")

baseline_plm_distilbert_base_uncased_campaign.json protocol: {'epochs': 4, 'batch_size': 16, 'lr': 5e-05, 'max_length': 128, 'early_stopping_patience': 0}
  epochs logged: 4
  f1_micro: 0.5819 ± 0.0012
  f1_macro: 0.4860 ± 0.0046
  exact_match: 0.4720 ± 0.0019
  map: 0.5057 ± 0.0002


In [6]:
import zipfile
from google.colab import files

def zip_backbone(slug: str, out_name: str) -> None:
    campaign = Path(f"reference/artifacts/baseline_plm_{slug}_campaign.json")
    if not campaign.is_file():
        raise FileNotFoundError(f"Run training first: {campaign}")
    zip_path = Path(f"/content/{out_name}")
    with zipfile.ZipFile(zip_path, "w", compression=zipfile.ZIP_DEFLATED) as zf:
        zf.write(campaign, campaign.name)
        for m in sorted(Path("runs").glob(f"{slug}_seed*_baseline_plm/metrics.json")):
            zf.write(m, f"{m.parent.name}/{m.name}")
    print(f"Download {out_name} ({zip_path.stat().st_size / 1e3:.1f} KB)")
    files.download(str(zip_path))

zip_backbone("distilbert_base_uncased", "baseline_plm_distilbert_step0.zip")

Download baseline_plm_distilbert_step0.zip (3.5 KB)


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

### Phase 2 — RoBERTa (uncomment the next cell, run it, then run the download cell below)

In [4]:
!./run_baseline_campaign.sh --backbone roberta-base {TRAIN_FLAGS}

config.json: 100% 481/481 [00:00<00:00, 1.89MB/s]
tokenizer_config.json: 100% 25.0/25.0 [00:00<00:00, 122kB/s]
vocab.json: 100% 899k/899k [00:00<00:00, 1.83MB/s]
merges.txt: 100% 456k/456k [00:00<00:00, 5.02MB/s]
tokenizer.json: 100% 1.36M/1.36M [00:00<00:00, 7.78MB/s]
README.md: 100% 9.40k/9.40k [00:00<00:00, 25.4MB/s]

simplified/train-00000-of-00001.parquet: downloading bytes:   0% 0.00/2.77M [00:00<?, ?B/s]
simplified/train-00000-of-00001.parquet: downloading bytes:  93% 2.59M/2.77M [00:01<00:00, 2.11MB/s]
simplified/train-00000-of-00001.parquet: downloading bytes: 100% 2.73M/2.73M [00:01<00:00, 2.04MB/s,  262kB/s  ]
simplified/train-00000-of-00001.parquet: reconstructing file: 100% 2.77M/2.77M [00:01<00:00, 2.07MB/s,  267kB/s  ]

simplified/validation-00000-of-00001.par(…): downloading bytes:   0% 0.00/350k [00:00<?, ?B/s]
simplified/validation-00000-of-00001.par(…): downloading bytes: 100% 346k/346k [00:01<00:00, 325kB/s, 33.6kB/s  ]
simplified/validation-00000-of-00001.par(…): r

In [10]:
import json
from pathlib import Path

def show_campaign(name: str) -> None:
    path = Path("reference/artifacts") / name
    if not path.is_file():
        print(f"Missing {name}")
        return
    c = json.loads(path.read_text(encoding="utf-8"))
    print(path.name, "protocol:", c.get("protocol", {}))
    print("  epochs logged:", len(c["runs"][0]["history"]))
    for k, b in c["test_aggregate"].items():
        print(f"  {k}: {b['mean']:.4f} ± {b['std']:.4f}")

show_campaign("baseline_plm_roberta_base_campaign.json")

baseline_plm_roberta_base_campaign.json protocol: {'epochs': 4, 'batch_size': 16, 'lr': 5e-05, 'max_length': 128, 'early_stopping_patience': 0}
  epochs logged: 4
  f1_micro: 0.5859 ± 0.0027
  f1_macro: 0.4871 ± 0.0048
  exact_match: 0.4729 ± 0.0019
  map: 0.5125 ± 0.0017


In [12]:
import zipfile
from google.colab import files

def zip_backbone(slug: str, out_name: str) -> None:
    campaign = Path(f"reference/artifacts/baseline_plm_{slug}_campaign.json")
    if not campaign.is_file():
        raise FileNotFoundError(f"Run training first: {campaign}")
    zip_path = Path(f"/content/{out_name}")
    with zipfile.ZipFile(zip_path, "w", compression=zipfile.ZIP_DEFLATED) as zf:
        zf.write(campaign, campaign.name)
        for m in sorted(Path("runs").glob(f"{slug}_seed*_baseline_plm/metrics.json")):
            zf.write(m, f"{m.parent.name}/{m.name}")
    print(f"Download {out_name} ({zip_path.stat().st_size / 1e3:.1f} KB)")
    files.download(str(zip_path))

zip_backbone("roberta_base", "baseline_plm_roberta_step0.zip")

Download baseline_plm_roberta_step0.zip (3.4 KB)


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>